---
---
# **Tutorial 1B:** *Predicting Patient Cost & Risk*
### *From a straight line, to a risk score, to AI that acts on it*
---
---

### QUESTION FOR TODAY
> *Can we look at simple **patient data** — age, BMI, smoking — and **predict healthcare cost** early, so care teams can step in before things get expensive?*

### TASK FOR TODAY

> 👤 **Meet John** — *45 years old, BMI 31, smoker*. We'll ask our models, one by one:

> *John, how much will it cost us to cover your healthcare this year?*

> *Are you someone our care team should call?*

### OUR ROADMAP
| Step | What we do | The question |
|---|---|---|
| 1 | **Linear Regression** | *How much* will John cost?|
| 2 | **Add more Features** | Which patient facts matter most? |
| 3 | **Logistic Regression** | Is John high-risk: YES or NO? |

### THE ONE IDEA BEHIND ALL OF IT
> **PREDICT → MEASURE THE ERROR → ADJUST → REPEAT.**

> *Every model today runs this same loop. Only the number of PARAMETERS changes.*


---
# **Dataset - *Medical Cost Personal* (insurance.csv)**
---
- *1,338 rows*
- *7 columns*
- *synthetic dataset.*

We load the dataset from [Kaggle](https://www.kaggle.com/datasets/mirichoi0218/insurance/data?select=insurance.csv) using [Github](https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/refs/heads/master/insurance.csv).

> ⚠️ Disclaimer: *This is practice data for learning purposes only. Do not use these models to make real medical or financial decisions. In the real world, handling real patient data requires strict privacy and security rules first.*

### Let's load the data !

In [ ]:
# ---------------------------------------------------------
# 1. BRINGING IN OUR TOOLKITS
# ---------------------------------------------------------
import pandas as pd           # For managing our data as a table
import numpy as np            # For math calculations
import matplotlib.pyplot as plt # For drawing charts and graphs

# ---------------------------------------------------------
# 2. BRINGING IN OUR AI/MATH ENGINES (From Scikit-Learn)
# ---------------------------------------------------------
# To split our dataset into a "study guide" (train) and a "final exam" (test)
from sklearn.model_selection import train_test_split

# The specific algorithms for Step 1 (How much?) and Step 3 (High risk: Yes/No?)
from sklearn.linear_model import LinearRegression, LogisticRegression

# The tools to "Measure the Error" and see how well our models guessed
from sklearn.metrics import mean_absolute_error, accuracy_score, confusion_matrix

# ---------------------------------------------------------
# 3. LOADING THE DATA
# ---------------------------------------------------------
# Download the CSV file directly from a public link
url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/refs/heads/master/insurance.csv"
insurance_data = pd.read_csv(url)

print(f"Successfully loaded {len(insurance_data)} members from insurance.csv file. \n")

# Peek at the first 5 rows to see what our patient data looks like
insurance_data.head()

---
# **Step 1: A straight line - *How much will John's medical care cost?***
---

Simplest possible start: *Does cost go up with age? Let's just plot it.*

### Let's plot *age* and *charges* on a straight line !

In [ ]:
# ---------------------------------------------------------
# DRAWING OUR DATA CLOUD
# ---------------------------------------------------------
# Set up the size of our blank canvas
plt.figure(figsize=(8, 4.5))

# Draw a dot for every single patient: Age on the bottom (x), Cost on the side (y)
# alpha=0.5 makes the dots slightly see-through so we can see where they overlap
plt.scatter(insurance_data["age"], insurance_data["charges"], alpha=0.5, color="blue", s=18)

# Label our chart so it is easy to read
plt.xlabel("Patient's Age")
plt.ylabel("Yearly Medical Charges ($)")
plt.title("Visualizing the Data: Older members tend to cost more")

# Show the final picture!
plt.show()

Clearly an upward trend!

***Linear regression*** just draws the best straight line through this cloud and once we have the line, we can read off a prediction for any age.

The plain-word math:

> **cost** ≈ (*the slope*) × **age** + (*the starting point*)

If you remember $y = mx + c$ from previous session, that is exactly what is happening here!

> The starting point ($c$): Where the line begins on the left side of our graph (the baseline medical cost).

> The slope ($m$): How steep the line is (how much the cost goes up for every single year a patient ages).

In ***machine learning***, these two numbers are our *adjustable parameters*.
The model's whole job is to adjust those two parameters until the line cuts perfectly through the middle of our data cloud.

Let's train it to find those parameters, and then ask it about John (age 45).

In [ ]:
# ---------------------------------------------------------
# TRAINING THE MODEL & PREDICTING FOR JOHN
# ---------------------------------------------------------
# Hold out 20% of members as a "final exam" to test the model later
# The model NEVER sees this test data while it is learning!
train, test = train_test_split(insurance_data, test_size=0.2, random_state=42)

# Train the engine: "Hey model, find the best straight line between Age and Charges"
straight_line = LinearRegression().fit(train[["age"]], train["charges"])

# Now that it has learned the line, let's ask it about John (age 45)
john_age = pd.DataFrame({"age": [45]})
john_pred_on_age_feature = straight_line.predict(john_age)[0]
#john_pred_on_age_feature = straight_line.predict(john_age)

print("John's COST Prediction based on the AGE feature")
john_pred_on_age_feature

In [ ]:
# ---------------------------------------------------------
# VISUALIZING THE LEARNED LINE & JOHN's PREDICTION
# ---------------------------------------------------------
plt.figure(figsize=(8, 4.5))

# 1. Draw the "test" patients the model is being graded on
plt.scatter(test["age"], test["charges"], alpha=0.4, color="blue", s=18)

# 2. Draw the actual line the model learned (from age 18 to 65)
ages = pd.DataFrame({"age": range(18, 65)})
plt.plot(ages["age"], straight_line.predict(ages), color="red", linewidth=2, label="Learned Line")

# 3. Put a giant star exactly where John sits on that line
plt.scatter([45], [john_pred_on_age_feature], color="black", s=120, zorder=5, marker="*", label="John (45)")

# 4. Clean up the chart labels
plt.xlabel("Patient's Age")
plt.ylabel("Yearly Medical Charges ($)")
plt.legend()
plt.title(f"The model learned: each year of age adds ≈ ${straight_line.coef_[0]:,.0f} to the cost")
plt.show()

In [ ]:
# ---------------------------------------------------------
# PREDICTING FOR JOHN (TAKE 1)
# ---------------------------------------------------------
# Print the final dollar amount prediction
print(f"Line's prediction for John (age 45): ${john_pred_on_age_feature:,.0f}")

So the age-only line guesses about $14,700 for John.

***But is age alone enough?***

*Let's find out how wrong this model is on average across all the members we held back:*

In [ ]:
# ---------------------------------------------------------
# MEASURING THE ERROR (How wrong were we?)
# ---------------------------------------------------------
# 1. Let's peek at 5 specific patients to see the mistakes firsthand
sample_patients = test.head(5).copy() # Grab the first 5 patients from our "final exam"

# Create a mini scorecard table
comparison = pd.DataFrame({
    "Age": sample_patients["age"],
    "Actual Bill ($)": sample_patients["charges"],
    "Model's Guess ($)": straight_line.predict(sample_patients[["age"]])
})

# Format the table so the numbers look like clean dollar amounts
pd.options.display.float_format = '${:,.0f}'.format

print("Side-by-Side Comparison (First 5 Patients):")
print(comparison)
print("\n" + "-"*50 + "\n") # Just draws a nice dividing line

# 2. Now calculate the average mistake across ALL test patients
miss_age = mean_absolute_error(test["charges"], straight_line.predict(test[["age"]]))

print(f"Average miss using AGE ALONE: ${miss_age:,.0f}")
print("That's a typical error. Age alone clearly isn't the whole story.")

---
# **Step 2: More Features - *Which patient facts matter most?***
---

Let's hand the model two more columns it's been missing:
> **BMI** and **smoking**. *(We turn smoker yes/no into 1/0 so the math can use it.)*

### ❓ Quick Audience Poll ------------------------------------
> Please type **A**, **B**, or **C**: **which do you think adds the *most* to yearly cost?**

> **A.**  each year of age <br>
**B.**  each BMI point <br>
**C.**  being a smoker

In [ ]:
# ---------------------------------------------------------
# ADDING NEW FEATURES & RETRAINING
# ---------------------------------------------------------
# Turn the text "yes" and "no" into 1 and 0 so the math engine can read it
insurance_data["smoker_flag"] = insurance_data["smoker"].map({"yes": 1, "no": 0})

# Split our data into a "study guide" (train) and "final exam" (test) again
train, test = train_test_split(insurance_data, test_size=0.2, random_state=42)

# Tell the model to look at three features instead of just one
FEATURES = ["age", "bmi", "smoker_flag"]

# Train the engine to find the best line using all three features
model_on_three_features = LinearRegression().fit(train[FEATURES], train["charges"])

In [ ]:
# ---------------------------------------------------------
# MEASURING THE NEW ERROR
# ---------------------------------------------------------
# Let's test the new model to see how much its guesses miss by on average
miss_full = mean_absolute_error(test["charges"], model_on_three_features.predict(test[FEATURES]))

print(f"Average miss with AGE ALONE   : ${miss_age:,.0f}")
print(f"Average miss with 3 FEATURES  : ${miss_full:,.0f}   <- much better\n")

In [ ]:
# ---------------------------------------------------------
# REVEALING THE POLL ANSWER: WHAT MATTERS MOST?
# ---------------------------------------------------------
print("What the model learned each FEATURE is 'worth' per year:")

# Let's peek at the model's new "parameters" to see how much cost it adds for each feature
for feat, weight in zip(["age", "bmi", "smoker"], model_on_three_features.coef_):
    print(f"   {feat:8s} -> +${weight:,.0f}")

> Poll answer: (c), and it's not close.

*Being a smoker adds about +$23,700 a year, roughly a hundred years of aging.*

### Watch what that does to John's prediction now that the model knows he smokes:

In [ ]:
# ---------------------------------------------------------
# PREDICTING FOR JOHN (TAKE 2)
# ---------------------------------------------------------
# We feed the model all of John's facts: Age 45, BMI 31, Smoker = 1 (Yes)
john = pd.DataFrame({"age": [45], "bmi": [31.0], "smoker_flag": [1]})

print(f"Age-only guess for John   : ${john_pred_on_age_feature:,.0f}")
print(f"Full model guess for John : ${model_on_three_features.predict(john)[0]:,.0f}   <- smoking changes everything")

print("\n(For comparison, real members just like John - 45, BMI~31, smokers - actually cost about $40,000. \n The model is now moving in the right direction.)")

***Takeaway:***
> *A model is only as smart as the **features** you give it. The big jump came from better information, not fancier math. We are using the exact same straight-line engine — just with more **parameters** (now **4**: one per feature, plus a starting point).*

---

<details><summary><b>Reveal the Math</b> (click)</summary>

> In Step 1, our straight-line engine only looked at age. It had exactly two parameters to learn (one for the feature, plus the starting point/intercept):
$$ \text{Cost} = (w_{\text{age}} \times \text{Age}) + \text{Intercept} $$


> In Step 2, we did not change the underlying math engine at all. We just expanded the equation to include our new features. Now, the model has to learn four parameters (three weights for our features, plus the starting point/intercept):
$$ \text{Cost} = (w_{\text{age}} \times \text{Age}) + (w_{\text{bmi}} \times \text{BMI}) + (w_{\text{smoker}} \times \text{Smoker}) + \text{Intercept} $$

</details>

### ❓ FAQ ------------------------------------

> Wait, what is the difference between adding a "***Feature***" and finding a "***Parameter***"?

<details><summary><b>Reveal answer</b> (click)</summary>

> ***Features***: These are the facts we choose to give the model. Age, BMI, and Smoking status are features. They are the raw information coming in from the real world.

> ***Parameters***: These are the internal numbers the model calculates and adjusts on its own. For example, the specific decision to add exactly \$23,700 for a smoker? That \$23,700 is a parameter.

> The simple rule: ***We provide the features. The model learns the parameters.***

</details>

---
# **Step 3: A Yes/No answer - *Is John high-risk?***
---

*Often a care team doesn't need the exact dollar figure — they need a flag:*
> Is this patient likely to be high-cost — Yes or No?

So we make a new column, **High_Risk**: 1 if a member's charges are over $15,000, else 0. Then we use ***logistic regression*** — the same line as before, with one twist: it squishes the answer into a probability between 0% and 100%.

The plain-word math:
> *score the patient with a straight line* → *squeeze that score into a 0–100% probability* → *flag them if the probability is high enough*

The squeeze is done by an S-shaped curve (called the sigmoid) — you might have seen this in our previous lesson. That's the only new idea here!

In [ ]:
# ---------------------------------------------------------
# 1. SETTING UP THE YES/NO GOAL
# ---------------------------------------------------------
# Create our new target column: 1 if costs > $15,000, 0 if not
insurance_data["High_Risk"] = (insurance_data["charges"] > 15000).astype(int)

# Split into "study guide" (train) and "final exam" (test) again
train, test = train_test_split(insurance_data, test_size=0.2, random_state=42)

print(f"High-risk members (>$15k): {insurance_data['High_Risk'].mean():.0%} (about 1 in 4)")

In [ ]:
# Peek at the first 5 rows to see what our patient data looks like
insurance_data.head()

In [ ]:
# ---------------------------------------------------------
# 2. TRAINING THE PROBABILITY MODEL (Logistic Regression)
# ---------------------------------------------------------
# We give it a max_iter of 2000 so the math engine has enough time to find the best fit
risk_model = LogisticRegression(max_iter=2000).fit(train[FEATURES], train["High_Risk"])

# Grade the model: what percentage of Yes/No answers did it get exactly right?
accuracy = accuracy_score(test["High_Risk"], risk_model.predict(test[FEATURES]))
print(f"Accuracy on held-out members: {accuracy:.0%}")

In [ ]:
# ---------------------------------------------------------
# 3. PREDICTING FOR JOHN
# ---------------------------------------------------------
# .predict_proba() gives us two numbers: [Chance of 0, Chance of 1]
# We add [0, 1] at the end to grab the "Chance of 1" (High Risk)
john_risk = risk_model.predict_proba(john)[0, 1]

print(f"\nJohn's high-risk probability: {john_risk:.0%} -> the care team should probably call John.")

The model gives John a **96% risk score**.

Notice it says 96%, not a flat "yes" — it hands you a *probability*, and you decide where to draw the line.
> That cutoff is a care-team decision, not a math one.

To see the kinds of mistakes the model makes on our test group,
> we use a **confusion matrix** — a simple ***2×2 scorecard of right vs. wrong***:

In [ ]:
# ---------------------------------------------------------
# 4. THE CONFUSION MATRIX (Grading the specific mistakes)
# ---------------------------------------------------------
# Compare actual final exam answers to our model's predictions
cm = confusion_matrix(test["High_Risk"], risk_model.predict(test[FEATURES]))

# --- PLOTTING THE SCORECARD ---
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.imshow(cm, cmap="Greens") # Colors the boxes green

# Plain-English labels for the 4 possible outcomes
labels = [["Correctly cleared\n(healthy, said healthy)", "False alarm\n(healthy, said risky)"],
          ["MISSED case\n(risky, said healthy)", "Correctly caught\n(risky, said risky)"]]

# Stamp the text and numbers into the 4 boxes
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{labels[i][j]}\n\n{cm[i, j]}", ha="center", va="center", fontsize=10)

# Clean up the chart axes
ax.set_xticks([0, 1]); ax.set_xticklabels(["Predicted\nNot-Risky", "Predicted\nRisky"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Actually\nNot-Risky", "Actually\nRisky"])
ax.set_title("Confusion Matrix: Where the model is right and wrong")

plt.tight_layout()
plt.show()

### The *Healthcare* Takeaway

In [ ]:
caught = cm[1, 1] # Bottom-right box
missed = cm[1, 0] # Bottom-left box

print(f"Caught {caught} high-risk members; missed {missed}. \n")
print("In healthcare, a MISSED case (bottom-left) usually matters more than a false alarm \n- so we'd tune the flag to catch more, even at the cost of extra calls.")

---
# **The Bridge: from *predicting* to *acting* — Agentic AI**

Everything so far **predicts**. It produces a number or a flag. But a prediction just sits there until a human reads it.
> **Agentic AI takes that prediction and connects it to real-world tools to automatically execute the next steps.**

| | **Predictive ML** *(what we just built)* | **Agentic AI** *(the application side)* |
|---|---|---|
| **What it does** | Analyzes data to forecast an outcome or flag a risk | *Plans and takes actions* based on that flag, using tools |
| **Industrial example** | "John has a **96%** chance of being high-cost this year." | Sees John's flag → checks his schedule → <br> drafts an email offering a telehealth screening → <br> updates his care pathway in the CRM → <br> **routes it to a nurse to approve** |

**The one-sentence version:**
> The model we built is the **brain** that spots the risk. **Agentic AI** uses the tools (email, scheduling, databases) to actually *do* something about John, with a human signing off on anything that matters.

# ***Lightning* Round**

In [ ]:
# =====================================================================
# THE LIGHTNING ROUND: New Data, New Features, Same Engine in 1 Cell
# =====================================================================
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression

# 1. CREATE A BRAND NEW DATASET (Mock Health Data)
# Let's use different features: Age, Blood Pressure, and Exercise Hours
mock_data = pd.DataFrame({
    "age":            [25, 45, 55, 65, 35, 50, 60, 40],
    "blood_pressure": [110, 130, 140, 150, 120, 135, 145, 125],
    "exercise_hours": [5,  2,  1,  0,  4,  3,  1,  2], # Hours per week
    "charges":        [3000, 12000, 18000, 25000, 5000, 14000, 22000, 8000]
})

# Create our Yes/No target flag (1 if charges > $15,000)
mock_data["High_Risk"] = (mock_data["charges"] > 15000).astype(int)
NEW_FEATURES = ["age", "blood_pressure", "exercise_hours"]

# 2. TRAIN BOTH ENGINES INSTANTLY
# We hand the exact same algorithms our new data table
cost_model = LinearRegression().fit(mock_data[NEW_FEATURES], mock_data["charges"])
risk_model = LogisticRegression().fit(mock_data[NEW_FEATURES], mock_data["High_Risk"])

# 3. MEET ROSE
# Rose is 60, has a BP of 145, and exercises 1 hour a week
rose = pd.DataFrame({"age": [60], "blood_pressure": [145], "exercise_hours": [1]})

# 4. MAKE PREDICTIONS FOR ROSE
rose_cost = cost_model.predict(rose)[0]
rose_risk = risk_model.predict_proba(rose)[0, 1] # Grab the probability of "1" (High Risk)

# 5. PRINT THE RESULTS
print("--- LIGHTNING ROUND: PREDICTING FOR ROSE ---")
print(f"Rose's Features: Age 60 | BP 145 | Exercise 1 hr/wk")
print(f"Predicted Cost:  ${rose_cost:,.0f}")
print(f"High-Risk Score: {rose_risk:.0%}")

---
# **Thank you !**
---

Author: *Aneetta Sara Shany*

Date: *2026 July 15, Wednesday*
